# 使用 Codex CLI 和 Agents SDK 构建一致的工作流

> **已归档示例——Codex MCP 服务器已弃用。** 本 notebook 使用 `codex mcp-server`，[当前文档](https://learn.chatgpt.com/docs/mcp-server)仅为现有集成保留了它。对于新的自动化和 CI 任务，请从 [Codex SDK](https://learn.chatgpt.com/docs/codex-sdk) 开始。如需管理身份验证、对话历史、审批和流式事件的更深度集成，请使用 [Codex app server](https://learn.chatgpt.com/docs/app-server)。这些是不同的接口；下面的代码并不是迁移示例。

### 确保智能体开发可重复、可追踪、可扩展

## 简介
开发者在做每件事时都力求一致。借助 Codex CLI 和 Agents SDK，这种一致性如今能够以前所未有的方式扩展。无论你是在重构大型代码库、推出新功能，还是引入新的测试框架，Codex 都能无缝集成到 CLI、IDE 和云端工作流中，自动化并强制执行可重复的开发模式。  

在本教程中，我们将使用 Agents SDK 构建单智能体和多智能体系统，并将 Codex CLI 作为 MCP 服务器对外提供。这将带来：  
- **一致性与可重复性**：为每个智能体提供限定范围的上下文。  
- **可扩展的编排**：协调单智能体和多智能体系统。  
- **可观测性与可审计性**：查看完整的智能体调用栈追踪。  

## 我们将介绍的内容
- 将 Codex CLI 初始化为 MCP 服务器：如何将 Codex 作为长期运行的 MCP 进程运行。  
- 构建单智能体系统：使用 Codex MCP 处理限定范围的任务。  
- 编排多智能体工作流：协调多个专门的智能体。  
- 追踪智能体行为：利用智能体追踪获得可见性并进行评估。  

## 前提条件与设置
开始本教程之前，请确保你具备以下条件：  
- 基本的编程基础：你应熟悉 Python 和 JavaScript。  
- 开发环境：你需要一个 IDE，例如 VS Code 或 Cursor。  
- OpenAI API 密钥：在 OpenAI Dashboard 中创建或找到你的 API 密钥。


## 环境设置
1. 在你的目录中创建一个 `.env` 文件夹，并添加你的 `OPENAI_API_KEY` 密钥
2. 安装依赖


In [ ]:
%pip install openai-agents openai ## install dependencies

## 将 Codex CLI 初始化为 MCP 服务器
这里我们在 Agents SDK 中将 Codex CLI 作为 MCP 服务器运行。我们提供 `codex mcp` 的初始化参数。该命令会将 Codex CLI 作为 MCP 服务器启动，并在 MCP 服务器上暴露两个 Codex 工具——`codex()` 和 `codex-reply()`。它们是 Agents SDK 需要调用 Codex 时所调用的底层工具。  
- `codex()` 用于创建对话。  
- `codex-reply()` 用于继续对话。  

```python
import asyncio
from agents import Agent, Runner
from agents.mcp import MCPServerStdio

async def main() -> None:
    async with MCPServerStdio(
        name="Codex CLI",
        params={
            "command": "npx",
            "args": ["-y", "codex", "mcp-server"],
        },
        client_session_timeout_seconds=360000,
    ) as codex_mcp_server:
        print("Codex MCP server started.")
        # We will add more code here in the next section
        return
```

另请注意，我们延长了 MCP 服务器的超时时间，以便 Codex CLI 有足够的时间执行并完成给定的任务。  

---

## 构建单智能体系统
我们从一个使用 Codex MCP 服务器的简单示例开始。我们定义两个智能体：  
1. **设计师智能体（Designer Agent）**：进行头脑风暴，为一款游戏撰写简短的设计简报。  
2. **开发者智能体（Developer Agent）**：根据设计师的规格实现一个简单的游戏。

```python
developer_agent = Agent(
    name="Game Developer",
    instructions=(
        "You are an expert in building simple games using basic html + css + javascript with no dependencies. "
        "Save your work in a file called index.html in the current directory."
        "Always call codex with \"approval-policy\": \"never\" and \"sandbox\": \"workspace-write\""
    ),
    mcp_servers=[codex_mcp_server],
)

designer_agent = Agent(
    name="Game Designer",
    instructions=(
        "You are an indie game connoisseur. Come up with an idea for a single page html + css + javascript game that a developer could build in about 50 lines of code. "
        "Format your request as a 3 sentence design brief for a game developer and call the Game Developer coder with your idea."
    ),
    model="gpt-5",
    handoffs=[developer_agent],
)

result = await Runner.run(designer_agent, "Implement a fun new game!")
```

注意，我们赋予了开发者智能体在不征求用户许可的情况下向项目目录写入文件的能力。  

现在运行代码，你会看到生成了一个 `index.html` 文件。打开该文件，开始玩游戏吧！ 

下面是我的智能体系统所创建游戏的几张截图。你的结果会有所不同！

| 游戏画面示例 | 游戏结束得分 |
| :---: | :---: |
| <img src="/images/game_example_1.png" alt="Example gameplay" width="320" /> | <img src="/images/game_example_2.png" alt="Game Over Score" width="320" /> |

下面是完整的可执行代码。注意，运行可能需要几分钟。如果你看到生成了 index.html 文件，就说明运行成功了。你可能还会看到一些关于格式的 MCP 事件警告，可以忽略这些事件。

In [ ]:
import os
from dotenv import load_dotenv
import asyncio
from agents import Agent, Runner, set_default_openai_api
from agents.mcp import MCPServerStdio

load_dotenv(override=True) # load the API key from the .env file. We set override to True here to ensure the notebook is loading any changes
set_default_openai_api(os.getenv("OPENAI_API_KEY"))

async def main() -> None:
    async with MCPServerStdio(
        name="Codex CLI",
        params={
            "command": "npx",
            "args": ["-y", "codex", "mcp-server"],
        },
        client_session_timeout_seconds=360000,
    ) as codex_mcp_server:
        developer_agent = Agent(
            name="Game Developer",
            instructions=(
                "You are an expert in building simple games using basic html + css + javascript with no dependencies. "
                "Save your work in a file called index.html in the current directory."
                "Always call codex with \"approval-policy\": \"never\" and \"sandbox\": \"workspace-write\""
            ),
            mcp_servers=[codex_mcp_server],
        )

        designer_agent = Agent(
            name="Game Designer",
            instructions=(
                "You are an indie game connoisseur. Come up with an idea for a single page html + css + javascript game that a developer could build in about 50 lines of code. "
                "Format your request as a 3 sentence design brief for a game developer and call the Game Developer coder with your idea."
            ),
            model="gpt-5",
            handoffs=[developer_agent],
        )

        result = await Runner.run(designer_agent, "Implement a fun new game!")
        # print(result.final_output)


if __name__ == "__main__":
    # Jupyter/IPython already runs an event loop, so calling asyncio.run() here
    # raises "asyncio.run() cannot be called from a running event loop".
    # Workaround: if a loop is running (notebook), use top-level `await`; otherwise use asyncio.run().
    try:
        asyncio.get_running_loop()
        await main()
    except RuntimeError:
        asyncio.run(main())

---

## 编排多智能体工作流
对于更大的工作流，我们引入一个智能体团队：  
- **项目经理（Project Manager）**：拆解任务列表、制定需求并协调工作。  
- **设计师（Designer）**：产出 UI/UX 规格。  
- **前端开发者（Frontend Developer）**：实现 UI/UX。  
- **后端开发者（Backend Developer）**：实现 API 和逻辑。  
- **测试员（Tester）**：根据验收标准验证产出。  

在本例中，我们有意让项目经理智能体在各个下游专门智能体之间执行门控逻辑。这确保在进行交接之前相应的产物已经存在。这与真实企业中的工作流相似，例如 JIRA 任务编排、长链条的发布流程以及 QA 签核。  

<div align="center">
  <img src="/images/multi_agent_codex_workflow.png" alt="Multi-Agent Codex Workflow with Codex MCP" style="max-width: 100%; width: 960px;" />
  <br />
  <em>使用 Codex MCP 和带门控的交接来产出产物的多智能体编排。</em>
</div>


在这种结构中，每个智能体都承担专门的职责。项目经理全面负责协调其他所有智能体，并确保整体任务完成。

## 定义 Codex CLI MCP 服务器
我们像在单智能体示例中一样设置 MCP 服务器来初始化 Codex CLI。

```python
async def main() -> None:
    async with MCPServerStdio(
        name="Codex CLI",
        params={
            "command": "npx",
            "args": ["-y", "codex", "mcp-server"],
        },
        client_session_timeout_seconds=360000,
    ) as codex_mcp_server:
        print("Codex MCP server started.")
        # We will add more code here in the next section
        return
 ```



## 定义各个专门智能体
下面我们定义每个专门智能体，并为它们提供对 Codex MCP 服务器的访问。注意，我们还向每个智能体传入了 `RECOMMENDED_PROMPT_PREFIX`，它有助于系统优化智能体之间的交接。  

```python
# Downstream agents are defined first for clarity, then PM references them in handoffs.
designer_agent = Agent(
    name="Designer",
    instructions=(
        f"""{RECOMMENDED_PROMPT_PREFIX}"""
        "You are the Designer.\n"
        "Your only source of truth is AGENT_TASKS.md and REQUIREMENTS.md from the Project Manager.\n"
        "Do not assume anything that is not written there.\n\n"
        "You may use the internet for additional guidance or research."
        "Deliverables (write to /design):\n"
        "- design_spec.md – a single page describing the UI/UX layout, main screens, and key visual notes as requested in AGENT_TASKS.md.\n"
        "- wireframe.md – a simple text or ASCII wireframe if specified.\n\n"
        "Keep the output short and implementation-friendly.\n"
        "When complete, handoff to the Project Manager with transfer_to_project_manager."
        "When creating files, call Codex MCP with {\"approval-policy\":\"never\",\"sandbox\":\"workspace-write\"}."
    ),
    model="gpt-5",
    tools=[WebSearchTool()],
    mcp_servers=[codex_mcp_server],
    handoffs=[],
)

frontend_developer_agent = Agent(
    name="Frontend Developer",
    instructions=(
        f"""{RECOMMENDED_PROMPT_PREFIX}"""
        "You are the Frontend Developer.\n"
        "Read AGENT_TASKS.md and design_spec.md. Implement exactly what is described there.\n\n"
        "Deliverables (write to /frontend):\n"
        "- index.html – main page structure\n"
        "- styles.css or inline styles if specified\n"
        "- main.js or game.js if specified\n\n"
        "Follow the Designer’s DOM structure and any integration points given by the Project Manager.\n"
        "Do not add features or branding beyond the provided documents.\n\n"
        "When complete, handoff to the Project Manager with transfer_to_project_manager_agent."
        "When creating files, call Codex MCP with {\"approval-policy\":\"never\",\"sandbox\":\"workspace-write\"}."
    ),
    model="gpt-5",
    mcp_servers=[codex_mcp_server],
    handoffs=[],
)

backend_developer_agent = Agent(
    name="Backend Developer",
    instructions=(
        f"""{RECOMMENDED_PROMPT_PREFIX}"""
        "You are the Backend Developer.\n"
        "Read AGENT_TASKS.md and REQUIREMENTS.md. Implement the backend endpoints described there.\n\n"
        "Deliverables (write to /backend):\n"
        "- package.json – include a start script if requested\n"
        "- server.js – implement the API endpoints and logic exactly as specified\n\n"
        "Keep the code as simple and readable as possible. No external database.\n\n"
        "When complete, handoff to the Project Manager with transfer_to_project_manager_agent."
        "When creating files, call Codex MCP with {\"approval-policy\":\"never\",\"sandbox\":\"workspace-write\"}."
    ),
    model="gpt-5",
    mcp_servers=[codex_mcp_server],
    handoffs=[],
)

tester_agent = Agent(
    name="Tester",
    instructions=(
        f"""{RECOMMENDED_PROMPT_PREFIX}"""
        "You are the Tester.\n"
        "Read AGENT_TASKS.md and TEST.md. Verify that the outputs of the other roles meet the acceptance criteria.\n\n"
        "Deliverables (write to /tests):\n"
        "- TEST_PLAN.md – bullet list of manual checks or automated steps as requested\n"
        "- test.sh or a simple automated script if specified\n\n"
        "Keep it minimal and easy to run.\n\n"
        "When complete, handoff to the Project Manager with transfer_to_project_manager."
        "When creating files, call Codex MCP with {\"approval-policy\":\"never\",\"sandbox\":\"workspace-write\"}."
    ),
    model="gpt-5",
    mcp_servers=[codex_mcp_server],
    handoffs=[],
)
```



每个角色完成其任务后，都会调用 `transfer_to_project_manager_agent`，由项目经理确认所需文件已存在（或要求修复），然后才放行下一个团队。  

## 定义项目经理智能体
项目经理是唯一接收初始提示词的智能体，它会在项目目录中创建规划文档，并在每次转交之前执行门控逻辑。  

```python        
project_manager_agent = Agent(
name="Project Manager",
instructions=(
    f"""{RECOMMENDED_PROMPT_PREFIX}"""
    """
    You are the Project Manager.

    Objective:
    Convert the input task list into three project-root files the team will execute against.

    Deliverables (write in project root):
    - REQUIREMENTS.md: concise summary of product goals, target users, key features, and constraints.
    - TEST.md: tasks with [Owner] tags (Designer, Frontend, Backend, Tester) and clear acceptance criteria.
    - AGENT_TASKS.md: one section per role containing:
        - Project name
        - Required deliverables (exact file names and purpose)
        - Key technical notes and constraints

    Process:
    - Resolve ambiguities with minimal, reasonable assumptions. Be specific so each role can act without guessing.
    - Create files using Codex MCP with {"approval-policy":"never","sandbox":"workspace-write"}.
    - Do not create folders. Only create REQUIREMENTS.md, TEST.md, AGENT_TASKS.md.

    Handoffs (gated by required files):
    1) After the three files above are created, hand off to the Designer with transfer_to_designer_agent and include REQUIREMENTS.md, and AGENT_TASKS.md.
    2) Wait for the Designer to produce /design/design_spec.md. Verify that file exists before proceeding.
    3) When design_spec.md exists, hand off in parallel to both:
        - Frontend Developer with transfer_to_frontend_developer_agent (provide design_spec.md, REQUIREMENTS.md, AGENT_TASKS.md).
        - Backend Developer with transfer_to_backend_developer_agent (provide REQUIREMENTS.md, AGENT_TASKS.md).
    4) Wait for Frontend to produce /frontend/index.html and Backend to produce /backend/server.js. Verify both files exist.
    5) When both exist, hand off to the Tester with transfer_to_tester_agent and provide all prior artifacts and outputs.
    6) Do not advance to the next handoff until the required files for that step are present. If something is missing, request the owning agent to supply it and re-check.

    PM Responsibilities:
    - Coordinate all roles, track file completion, and enforce the above gating checks.
    - Do NOT respond with status updates. Just handoff to the next agent until the project is complete.
    """
),
model="gpt-5",
model_settings=ModelSettings(
    reasoning=Reasoning(effort="medium")
),
handoffs=[designer_agent, frontend_developer_agent, backend_developer_agent, tester_agent],
mcp_servers=[codex_mcp_server],
)
```

构建好项目经理后，脚本会将每个专门智能体的交接目标都设置回项目
经理。这确保交付物在进入下一步之前会被送回进行验证。

```python
designer_agent.handoffs = [project_manager_agent]
frontend_developer_agent.handoffs = [project_manager_agent]
backend_developer_agent.handoffs = [project_manager_agent]
tester_agent.handoffs = [project_manager_agent]
```
## 添加任务列表
这是项目经理将为整个系统细化为具体需求和任务的任务。

```python
task_list = """
Goal: Build a tiny browser game to showcase a multi-agent workflow.

High-level requirements:
- Single-screen game called "Bug Busters".
- Player clicks a moving bug to earn points.
- Game ends after 20 seconds and shows final score.
- Optional: submit score to a simple backend and display a top-10 leaderboard.

Roles:
- Designer: create a one-page UI/UX spec and basic wireframe.
- Frontend Developer: implement the page and game logic.
- Backend Developer: implement a minimal API (GET /health, GET/POST /scores).
- Tester: write a quick test plan and a simple script to verify core routes.

Constraints:
- No external database—memory storage is fine.
- Keep everything readable for beginners; no frameworks required.
- All outputs should be small files saved in clearly named folders.
"""
```

接下来，运行你的系统，然后静候片刻，你会看到智能体们开始工作，并在几分钟内创建出一款游戏！我们在下方提供了完整的可执行代码。运行完成后，你会看到创建了如下文件目录。注意，这个多智能体编排通常需要大约 11 分钟才能完全完成。

```markdown
root_directory/
├── AGENT_TASKS.md
├── REQUIREMENTS.md
├── backend
│   ├── package.json
│   └── server.js
├── design
│   ├── design_spec.md
│   └── wireframe.md
├── frontend
│   ├── game.js
│   ├── index.html
│   └── styles.css
└── TEST.md
```

使用 `node server.js` 启动后端服务器，然后打开 `index.html` 文件即可开始玩游戏。


In [ ]:
import os
from dotenv import load_dotenv
import asyncio
from agents import Agent, Runner, WebSearchTool, ModelSettings, set_default_openai_api
from agents.mcp import MCPServerStdio
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from openai.types.shared import Reasoning

load_dotenv(override=True) # load the API key from the .env file. We set override to True here to ensure the notebook is loading any changes
set_default_openai_api(os.getenv("OPENAI_API_KEY"))

async def main() -> None:
    async with MCPServerStdio(
        name="Codex CLI",
        params={"command": "npx", "args": ["-y", "codex", "mcp-server"]},
        client_session_timeout_seconds=360000,
    ) as codex_mcp_server:

        # Downstream agents are defined first for clarity, then PM references them in handoffs.
        designer_agent = Agent(
            name="Designer",
            instructions=(
                f"""{RECOMMENDED_PROMPT_PREFIX}"""
                "You are the Designer.\n"
                "Your only source of truth is AGENT_TASKS.md and REQUIREMENTS.md from the Project Manager.\n"
                "Do not assume anything that is not written there.\n\n"
                "You may use the internet for additional guidance or research."
                "Deliverables (write to /design):\n"
                "- design_spec.md – a single page describing the UI/UX layout, main screens, and key visual notes as requested in AGENT_TASKS.md.\n"
                "- wireframe.md – a simple text or ASCII wireframe if specified.\n\n"
                "Keep the output short and implementation-friendly.\n"
                "When complete, handoff to the Project Manager with transfer_to_project_manager."
                "When creating files, call Codex MCP with {\"approval-policy\":\"never\",\"sandbox\":\"workspace-write\"}."
            ),
            model="gpt-5",
            tools=[WebSearchTool()],
            mcp_servers=[codex_mcp_server],
            handoffs=[],
        )

        frontend_developer_agent = Agent(
            name="Frontend Developer",
            instructions=(
                f"""{RECOMMENDED_PROMPT_PREFIX}"""
                "You are the Frontend Developer.\n"
                "Read AGENT_TASKS.md and design_spec.md. Implement exactly what is described there.\n\n"
                "Deliverables (write to /frontend):\n"
                "- index.html – main page structure\n"
                "- styles.css or inline styles if specified\n"
                "- main.js or game.js if specified\n\n"
                "Follow the Designer’s DOM structure and any integration points given by the Project Manager.\n"
                "Do not add features or branding beyond the provided documents.\n\n"
                "When complete, handoff to the Project Manager with transfer_to_project_manager_agent."
                "When creating files, call Codex MCP with {\"approval-policy\":\"never\",\"sandbox\":\"workspace-write\"}."
            ),
            model="gpt-5",
            mcp_servers=[codex_mcp_server],
            handoffs=[],
        )

        backend_developer_agent = Agent(
            name="Backend Developer",
            instructions=(
                f"""{RECOMMENDED_PROMPT_PREFIX}"""
                "You are the Backend Developer.\n"
                "Read AGENT_TASKS.md and REQUIREMENTS.md. Implement the backend endpoints described there.\n\n"
                "Deliverables (write to /backend):\n"
                "- package.json – include a start script if requested\n"
                "- server.js – implement the API endpoints and logic exactly as specified\n\n"
                "Keep the code as simple and readable as possible. No external database.\n\n"
                "When complete, handoff to the Project Manager with transfer_to_project_manager_agent."
                "When creating files, call Codex MCP with {\"approval-policy\":\"never\",\"sandbox\":\"workspace-write\"}."
            ),
            model="gpt-5",
            mcp_servers=[codex_mcp_server],
            handoffs=[],
        )

        tester_agent = Agent(
            name="Tester",
            instructions=(
                f"""{RECOMMENDED_PROMPT_PREFIX}"""
                "You are the Tester.\n"
                "Read AGENT_TASKS.md and TEST.md. Verify that the outputs of the other roles meet the acceptance criteria.\n\n"
                "Deliverables (write to /tests):\n"
                "- TEST_PLAN.md – bullet list of manual checks or automated steps as requested\n"
                "- test.sh or a simple automated script if specified\n\n"
                "Keep it minimal and easy to run.\n\n"
                "When complete, handoff to the Project Manager with transfer_to_project_manager."
                "When creating files, call Codex MCP with {\"approval-policy\":\"never\",\"sandbox\":\"workspace-write\"}."
            ),
            model="gpt-5",
            mcp_servers=[codex_mcp_server],
            handoffs=[],
        )

        project_manager_agent = Agent(
            name="Project Manager",
            instructions=(
                f"""{RECOMMENDED_PROMPT_PREFIX}"""
                """
                You are the Project Manager.

                Objective:
                Convert the input task list into three project-root files the team will execute against.

                Deliverables (write in project root):
                - REQUIREMENTS.md: concise summary of product goals, target users, key features, and constraints.
                - TEST.md: tasks with [Owner] tags (Designer, Frontend, Backend, Tester) and clear acceptance criteria.
                - AGENT_TASKS.md: one section per role containing:
                  - Project name
                  - Required deliverables (exact file names and purpose)
                  - Key technical notes and constraints

                Process:
                - Resolve ambiguities with minimal, reasonable assumptions. Be specific so each role can act without guessing.
                - Create files using Codex MCP with {"approval-policy":"never","sandbox":"workspace-write"}.
                - Do not create folders. Only create REQUIREMENTS.md, TEST.md, AGENT_TASKS.md.

                Handoffs (gated by required files):
                1) After the three files above are created, hand off to the Designer with transfer_to_designer_agent and include REQUIREMENTS.md, and AGENT_TASKS.md.
                2) Wait for the Designer to produce /design/design_spec.md. Verify that file exists before proceeding.
                3) When design_spec.md exists, hand off in parallel to both:
                   - Frontend Developer with transfer_to_frontend_developer_agent (provide design_spec.md, REQUIREMENTS.md, AGENT_TASKS.md).
                   - Backend Developer with transfer_to_backend_developer_agent (provide REQUIREMENTS.md, AGENT_TASKS.md).
                4) Wait for Frontend to produce /frontend/index.html and Backend to produce /backend/server.js. Verify both files exist.
                5) When both exist, hand off to the Tester with transfer_to_tester_agent and provide all prior artifacts and outputs.
                6) Do not advance to the next handoff until the required files for that step are present. If something is missing, request the owning agent to supply it and re-check.

                PM Responsibilities:
                - Coordinate all roles, track file completion, and enforce the above gating checks.
                - Do NOT respond with status updates. Just handoff to the next agent until the project is complete.
                """
            ),
            model="gpt-5",
            model_settings=ModelSettings(
                reasoning=Reasoning(effort="medium")
            ),
            handoffs=[designer_agent, frontend_developer_agent, backend_developer_agent, tester_agent],
            mcp_servers=[codex_mcp_server],
        )

        designer_agent.handoffs = [project_manager_agent]
        frontend_developer_agent.handoffs = [project_manager_agent]
        backend_developer_agent.handoffs = [project_manager_agent]
        tester_agent.handoffs = [project_manager_agent]

        # Example task list input for the Project Manager
        task_list = """
Goal: Build a tiny browser game to showcase a multi-agent workflow.

High-level requirements:
- Single-screen game called "Bug Busters".
- Player clicks a moving bug to earn points.
- Game ends after 20 seconds and shows final score.
- Optional: submit score to a simple backend and display a top-10 leaderboard.

Roles:
- Designer: create a one-page UI/UX spec and basic wireframe.
- Frontend Developer: implement the page and game logic.
- Backend Developer: implement a minimal API (GET /health, GET/POST /scores).
- Tester: write a quick test plan and a simple script to verify core routes.

Constraints:
- No external database—memory storage is fine.
- Keep everything readable for beginners; no frameworks required.
- All outputs should be small files saved in clearly named folders.
"""

        # Only the Project Manager receives the task list directly
        result = await Runner.run(project_manager_agent, task_list, max_turns=30)
        print(result.final_output)

if __name__ == "__main__":
    # Jupyter/IPython already runs an event loop, so calling asyncio.run() here
    # raises "asyncio.run() cannot be called from a running event loop".
    # Workaround: if a loop is running (notebook), use top-level `await`; otherwise use asyncio.run().
    try:
        asyncio.get_running_loop()
        await main()
    except RuntimeError:
        asyncio.run(main())

---

## 使用 Traces 追踪智能体行为
随着智能体系统复杂度的增加，了解这些智能体之间如何交互变得非常重要。我们可以借助 Traces 仪表盘做到这一点，它会记录：  
- 提示词、工具调用以及智能体之间的交接。  
- MCP 服务器调用、Codex CLI 调用、执行时间和文件写入。  
- 错误和警告。  

我们来看看上面这个智能体团队的追踪记录。

<div align="center">
  <img src="/images/multi_agent_trace.png" alt="Multi-Agent Codex Workflow with Codex MCP" style="max-width: 100%; width: 960px;" />
</div>

在这条追踪中，我们可以确认每次智能体交接都由项目经理智能体统筹指挥，它会在交接给下一个智能体之前确认特定产物已经存在。此外，我们还可以看到 Codex MCP 服务器的具体调用，以及通过调用 Responses API 生成的每个输出。时间线条突出显示了执行时长，便于发现耗时较长的步骤，并了解控制权如何在智能体之间传递。

你甚至可以点进每条追踪，查看提示词、工具调用和其他元数据的具体细节。随着时间推移，你可以借助这些信息进一步调优、优化和跟踪智能体系统的性能。

<div align="center">
  <img src="/images/multi_agent_trace_details.png" alt="Multi-Agent Trace Details" style="max-width: 100%; width: 960px;" />
</div>


---

## 本指南内容回顾
在本指南中，我们演示了使用 Codex CLI 和 Agents SDK 构建一致、可扩展工作流的过程。具体来说，我们介绍了：  

- **Codex MCP 服务器设置**：如何将 Codex CLI 初始化为 MCP 服务器，并将其作为工具提供给智能体交互使用。  
- **单智能体示例**：一个包含设计师智能体和开发者智能体的简单工作流，Codex 以确定性的方式执行限定范围的任务，产出一款可玩的游戏。  
- **多智能体编排**：扩展为包含项目经理、设计师、前端开发者、后端开发者和测试员的更大工作流，模拟复杂的任务编排和签核流程。  
- **Traces 与可观测性**：使用内置的 Traces 捕获提示词、工具调用、交接、执行时间和产物，全面了解智能体行为，便于调试、评估和后续优化。  

---

## 下一步：应用这些经验
现在你已经看到了 Codex MCP 和 Agents SDK 的实际效果，下面介绍如何在真实项目中应用这些概念并获取价值：  

### 1. 扩展到真实世界的大规模推行
- 将同样的多智能体编排应用于大型代码重构（例如 500 多个文件、框架迁移）。  
- 利用 Codex MCP 的确定性执行，实现长期运行、可审计且进度可追踪的推行。  

### 2. 在不失控的前提下加速交付
- 组织专门智能体团队并行开发，同时保留用于产物验证的门控逻辑。  
- 缩短新功能、测试或代码库现代化的周转时间。  

### 3. 扩展并连接到你的开发工作流
- 通过 webhook 将由 MCP 驱动的智能体与 Jira、GitHub 或 CI/CD 流水线连接起来，实现自动化、可重复的开发周期。  
- 在多智能体服务编排中利用 Codex MCP：不仅用于代码生成，还可用于文档、QA 和部署。  
